In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import re
from collections import Counter

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cuda


In [2]:
DATA_PATH = r"C:\Users\nihal\Desktop\NLP_Project\training_pairs.csv"

df = pd.read_csv(DATA_PATH)
print("Total training pairs:", len(df))

Total training pairs: 130790


In [3]:
# -------------------------
# Tokenization + vocab
# -------------------------

MAX_LEN = 60
VOCAB_SIZE = 40000

def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

counter = Counter()

for col in ["query", "positive_passage", "negative_passage"]:
    for txt in df[col].astype(str):
        counter.update(tokenize(txt))

vocab = {"<pad>": 0, "<unk>": 1}
for word, _ in counter.most_common(VOCAB_SIZE - 2):
    vocab[word] = len(vocab)

torch.save(vocab, r"C:\Users\nihal\Desktop\NLP_Project\bigru_vocab.pt")

def encode(text):
    tokens = tokenize(text)
    ids = [vocab.get(t, 1) for t in tokens][:MAX_LEN]
    return ids + [0] * (MAX_LEN - len(ids))

In [4]:
# -------------------------
# Dataset
# -------------------------

class RankingDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        return (
            torch.tensor(encode(row["query"])),
            torch.tensor(encode(row["positive_passage"])),
            torch.tensor(encode(row["negative_passage"]))
        )

In [5]:
# -------------------------
# STRONG BiGRU Model
# -------------------------

class BiGRURanker(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, 200)
        self.embed_dropout = nn.Dropout(0.3)

        self.gru = nn.GRU(
            200,
            256,
            batch_first=True,
            bidirectional=True,
            num_layers=2,
            dropout=0.3
        )

        self.layer_norm = nn.LayerNorm(512)

        self.fc = nn.Sequential(
            nn.Linear(512 * 4, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def encode(self, x):
        emb = self.embedding(x)
        emb = self.embed_dropout(emb)

        _, h = self.gru(emb)

        fwd = h[-2]
        bwd = h[-1]

        rep = torch.cat([fwd, bwd], dim=1)
        return self.layer_norm(rep)

    def forward(self, q, p):
        qv = self.encode(q)
        pv = self.encode(p)

        features = torch.cat([
            qv,
            pv,
            torch.abs(qv - pv),
            qv * pv
        ], dim=1)

        return self.fc(features).squeeze()

In [6]:
# -------------------------
# Training setup
# -------------------------

TRAIN_SIZE = min(60000, len(df))
BATCH_SIZE = 128
EPOCHS = 10
LR = 2e-4

train_df = df.sample(TRAIN_SIZE, random_state=42)
dataset = RankingDataset(train_df)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

model = BiGRURanker(len(vocab)).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS
)

def ranking_loss(pos, neg):
    return torch.mean(F.softplus(-(pos - neg)))

print("\nStrong BiGRU training started...\n")

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for q, pos, neg in tqdm(loader, desc=f"Epoch {epoch+1}"):

        q = q.to(DEVICE)
        pos = pos.to(DEVICE)
        neg = neg.to(DEVICE)

        pos_score = model(q, pos)
        neg_score = model(q, neg)

        loss = ranking_loss(pos_score, neg_score)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    scheduler.step()

    print(f"Epoch {epoch+1} avg loss: {total_loss / len(loader):.4f}")


Strong BiGRU training started...



Epoch 1: 100%|██████████| 468/468 [01:04<00:00,  7.29it/s]


Epoch 1 avg loss: 0.5559


Epoch 2: 100%|██████████| 468/468 [01:04<00:00,  7.26it/s]


Epoch 2 avg loss: 0.4495


...

Epoch 8 avg loss: 0.2581


Epoch 9: 100%|██████████| 468/468 [01:03<00:00,  7.34it/s]


Epoch 9 avg loss: 0.2500


Epoch 10: 100%|██████████| 468/468 [01:03<00:00,  7.33it/s]

Epoch 10 avg loss: 0.2433


In [7]:
MODEL_PATH = r"C:\Users\nihal\Desktop\NLP_Project\bigru_model.pt"
torch.save(model.state_dict(), MODEL_PATH)

print("\nStrong BiGRU model saved to:", MODEL_PATH)


Strong BiGRU model saved to: C:\Users\nihal\Desktop\NLP_Project\bigru_model.pt
